<h1>Librerías</h1>

In [88]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import os
import scipy.signal as sg
from concurrent.futures import ThreadPoolExecutor
%matplotlib inline
import pywt
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import librosa
from scipy.stats import entropy
import nolds
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


<h1>Funciones básicas</h1>

In [52]:
# Función de filtro pasa-banda utilizando el diseño de filtro Butterworth.
# Filtra los datos entre las frecuencias lowc y high.
def butter_bandpass_filter(senal: np.array, lowcut: float, highcut: float, fs: float, order: int):
    nyquist = 0.5 * fs  # Frecuencia de Nyquist, la mitad de la tasa de muestreo
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = sg.butter(order, [low, high], btype='band', analog=False)  # Diseña el filtro pasa-banda Butterworth
    y = sg.filtfilt(b, a, senal)  # Aplica el filtro a los datos usando filtrado cero-fase
    return y

In [53]:
def notch_filter(senal, sr, freq, quality):

    # Diseñar el filtro notch
    b, a = sg.iirnotch(freq, quality, sr)

    # Aplicar el filtro a la señal
    senal_filtrada = sg.lfilter(b, a, senal)

    return senal_filtrada

In [54]:
# Extraer MFCCs 

def extraer_caracteristicas_mfccs(audio, frecuencia, numero):
    mfccs = librosa.feature.mfcc(y=audio, sr=frecuencia, n_mfcc=numero)  #n:mfcc a evaluar según rendimiento del modelo

    mfccs_mean = np.mean(mfccs, axis=1)
    mfccs_std = np.std(mfccs, axis=1)
    mfccs_max = np.max(mfccs, axis=1)
    mfccs_min = np.min(mfccs, axis=1)


    # Concatenar estadísticas de MFCCs
    caracteristicas_mfcc = np.concatenate([mfccs_mean, mfccs_std, mfccs_max, mfccs_min])
    return caracteristicas_mfcc



In [55]:
# Función para extraer características wavelet
def extraer_caracteristicas_wavelet(senal, wavelet='db4', nivel=5):
    coeficientes = pywt.wavedec(senal, wavelet, level=nivel)
    caracteristicas_wavelet = []
    for coef in coeficientes:
        caracteristicas_wavelet.append(np.mean(coef))  # Media
        caracteristicas_wavelet.append(np.std(coef))   # Desviación estándar
        caracteristicas_wavelet.append(np.max(coef))   # Máximo
        caracteristicas_wavelet.append(np.min(coef))   # Mínimo
    return np.array(caracteristicas_wavelet)


In [56]:
def calcular_entropia(senal):
    hist, _ = np.histogram(senal, bins=50, density=True)
    hist = hist / np.sum(hist)
    return entropy(hist)

In [57]:
def extract_respiratory_features(audio, sr):
    """Extrae todas las características relevantes para respiración."""
    
    mfcc_features = extraer_caracteristicas_mfccs(audio, sr, 10)
    
    # 3. Características wavelet
    wavelet_features = extraer_caracteristicas_wavelet(audio)

    # Calcular características de complejidad
    entropia_shannon = calcular_entropia(audio)
    dimension_fractal = nolds.hurst_rs(audio)
    variabilidad = np.std(np.diff(audio))

    # Combinar todas las características
    caracteristicas = np.concatenate([mfcc_features, wavelet_features, [entropia_shannon, dimension_fractal, variabilidad]])

    # Combinar todas las características
    return caracteristicas

In [58]:
# Ruta de la carpeta donde están los archivos de audio
folder_path = "./Respiratory_Sound_Database\Respiratory_Sound_Database//audio_and_txt_files"
# Obtener lista de archivos de audio en la carpeta
audio_files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]


patients = pd.read_csv("Respiratory_Sound_Database\Respiratory_Sound_Database\patient_diagnosis.csv",  header=None)

In [59]:
def process_file(idx):
    # 1. Validación de archivo
    file_name = audio_files[idx]
    audio_path = os.path.join(folder_path, file_name)
    
    if not os.path.exists(audio_path):
        print(f"Archivo no encontrado: {audio_path}")
        return None

    try:
        # 2. Cargar con SR original y re-muestrear
        audio, sr = librosa.load(audio_path, sr=None)
        #audio, sr = resample_audio(audio, orig_sr, 22050)
        
        # 3. Validar duración después de re-muestreo
        if len(audio) < 0.5 * sr:  # Mínimo 0.5 segundos
            print(f"Audio demasiado corto: {file_name}")
            return None

        # 4. Obtener etiqueta (sick)
        patient_id = int(file_name[:3])
        sick = patients.loc[patients[0] == patient_id, 1].values[0]
        
    except Exception as e:
        print(f"Error cargando {file_name}: {str(e)}")
        return None
    
    # 5. Preprocesamiento de señal
    audio = butter_bandpass_filter(audio, 100, 1800, sr, order=4)
    audio = notch_filter(audio, sr, freq=50, quality=35)
    
    # 6. Extracción de características para TODO el audio
    features = extract_respiratory_features(audio, sr)

    
    return (features, sick)


In [60]:
uno, dos = process_file(919)

In [61]:
uno

array([-6.93517984e+02,  1.55312659e+02,  7.44960667e+01,  1.09618923e+01,
       -1.58528938e+00,  1.93775262e+01,  3.38326002e+01,  2.30735475e+01,
        1.04058807e+00, -1.05124615e+01,  2.51573811e+01,  2.98020884e+01,
        2.11274816e+01,  1.67434040e+01,  1.08599062e+01,  5.86928671e+00,
        6.76697137e+00,  8.46041632e+00,  9.47830813e+00,  8.74359576e+00,
       -5.68690973e+02,  2.72348296e+02,  1.26681224e+02,  4.69686628e+01,
        2.08312563e+01,  3.61738422e+01,  5.54856188e+01,  4.03841836e+01,
        2.01660456e+01,  5.12303569e+00, -7.30265796e+02,  1.09976744e+02,
        1.71606763e+01, -6.52474994e+01, -5.30676489e+01, -1.26970991e+01,
        1.12240812e+01,  1.34717319e+00, -2.36464281e+01, -3.45464346e+01,
       -3.56294366e-06,  2.64630378e-02,  2.75136076e-01, -3.15131473e-01,
       -8.02522155e-07,  1.99738386e-03,  6.32776789e-02, -6.96157731e-02,
        9.98317578e-09,  7.11821848e-04,  5.57100476e-02, -5.47547178e-02,
        7.09774296e-09,  

In [62]:
dos

'Pneumonia'

In [63]:
'''
# Inicializa diccionarios para almacenar los datos de entrenamiento y validación.
training_data = {
    'data': [],
    'label': []
}

validation_data = {
    'data': [],
    'label': []
}
'''

"\n# Inicializa diccionarios para almacenar los datos de entrenamiento y validación.\ntraining_data = {\n    'data': [],\n    'label': []\n}\n\nvalidation_data = {\n    'data': [],\n    'label': []\n}\n"

In [64]:
indexs = list(range(0, 920))

In [65]:
''''
# Primero, dividimos en entrenamiento + prueba
train_val, test = train_test_split(indexs, test_size=0.15, random_state=42)

# Luego, dividimos entrenamiento + validación
indexs_train, indexs_validation = train_test_split(train_val, test_size=0.176, random_state=42)  # 0.176 para que validación sea aprox 15%
'''

"'\n# Primero, dividimos en entrenamiento + prueba\ntrain_val, test = train_test_split(indexs, test_size=0.15, random_state=42)\n\n# Luego, dividimos entrenamiento + validación\nindexs_train, indexs_validation = train_test_split(train_val, test_size=0.176, random_state=42)  # 0.176 para que validación sea aprox 15%\n"

In [66]:
'''
# Procesar los archivos de entrenamiento en paralelo utilizando múltiples hilos (threads).
# Se usa `ThreadPoolExecutor` para paralelizar el procesamiento y mejorar la velocidad.
with ThreadPoolExecutor(max_workers=os.cpu_count() - 2) as executor:
    # Se aplica la función `process_file_moon` a cada índice en `indexs_train` y se muestra un progreso con tqdm.
    results = list(tqdm(executor.map(process_file, indexs_train), total=len(indexs_train)))

# Recopilar los datos procesados de entrenamiento.
for r in results:
    # Si el archivo no pudo procesarse (devuelve `None`), se imprime "skip" y se pasa al siguiente.
    if r[0] is None:
        print('skip')
        continue
    # Se agrega el dato procesado (entrada) a la lista `training_data['data']`.
    training_data['data'].append(r[0])
    # Se agrega la etiqueta correspondiente (salida) a la lista `training_data['label']`.
    training_data['label'].append(r[1])

# Procesar los archivos de validación en paralelo utilizando múltiples hilos (threads).
with ThreadPoolExecutor(max_workers=os.cpu_count() - 2) as executor:
    # Similar al procesamiento de entrenamiento, pero aplicado a los índices de validación.
    results = list(tqdm(executor.map(process_file, indexs_validation), total=len(indexs_validation)))

# Recopilar los datos procesados de validación.
for r in results:
    # Si el archivo no pudo procesarse, se imprime "skip" y se pasa al siguiente.
    if r[0] is None:
        print('skip')
        continue
    # Se agrega el dato procesado (entrada) a la lista `validation_data['data']`.
    validation_data['data'].append(r[0])
    # Se agrega la etiqueta correspondiente (salida) a la lista `validation_data['label']`.
    validation_data['label'].append(r[1])

# Mostrar las dimensiones del primer dato de entrada procesado.
inp, out = results[0]  # Se obtiene el primer par de entrada y salida de los resultados procesados.
print(inp.shape)       # Se imprime la forma (shape) de la entrada procesada para verificar sus dimensiones.
'''

'\n# Procesar los archivos de entrenamiento en paralelo utilizando múltiples hilos (threads).\n# Se usa `ThreadPoolExecutor` para paralelizar el procesamiento y mejorar la velocidad.\nwith ThreadPoolExecutor(max_workers=os.cpu_count() - 2) as executor:\n    # Se aplica la función `process_file_moon` a cada índice en `indexs_train` y se muestra un progreso con tqdm.\n    results = list(tqdm(executor.map(process_file, indexs_train), total=len(indexs_train)))\n\n# Recopilar los datos procesados de entrenamiento.\nfor r in results:\n    # Si el archivo no pudo procesarse (devuelve `None`), se imprime "skip" y se pasa al siguiente.\n    if r[0] is None:\n        print(\'skip\')\n        continue\n    # Se agrega el dato procesado (entrada) a la lista `training_data[\'data\']`.\n    training_data[\'data\'].append(r[0])\n    # Se agrega la etiqueta correspondiente (salida) a la lista `training_data[\'label\']`.\n    training_data[\'label\'].append(r[1])\n\n# Procesar los archivos de validació

In [67]:
'''
# Convertir las listas en arreglos de numpy:

training_data['data'] = np.array(training_data['data'])  # Convierte los datos de entrenamiento en un arreglo numpy
training_data['label'] = np.array(training_data['label'])  # Convierte las etiquetas de entrenamiento en un arreglo numpy

validation_data['data'] = np.array(validation_data['data'])  # Convierte los datos de validación en un arreglo numpy
validation_data['label'] = np.array(validation_data['label'])  # Convierte las etiquetas de validación en un arreglo numpy
'''

"\n# Convertir las listas en arreglos de numpy:\n\ntraining_data['data'] = np.array(training_data['data'])  # Convierte los datos de entrenamiento en un arreglo numpy\ntraining_data['label'] = np.array(training_data['label'])  # Convierte las etiquetas de entrenamiento en un arreglo numpy\n\nvalidation_data['data'] = np.array(validation_data['data'])  # Convierte los datos de validación en un arreglo numpy\nvalidation_data['label'] = np.array(validation_data['label'])  # Convierte las etiquetas de validación en un arreglo numpy\n"

In [68]:
# print(training_data['label'].shape)


In [69]:
# print(validation_data['data'].shape)

In [70]:
'''
X_train = training_data['data']
Y_train = training_data['label']

X_eval = validation_data['data']
Y_eval = validation_data['label']
'''

"\nX_train = training_data['data']\nY_train = training_data['label']\n\nX_eval = validation_data['data']\nY_eval = validation_data['label']\n"

In [71]:
'''

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Ajustanos mean y std

X_val_scaled = scaler.transform(X_eval)    #Con los datos de train
#X_test_scaled = scaler.transform(X_test)
'''

'\n\nscaler = StandardScaler()\nX_train_scaled = scaler.fit_transform(X_train)  # Ajustanos mean y std\n\nX_val_scaled = scaler.transform(X_eval)    #Con los datos de train\n#X_test_scaled = scaler.transform(X_test)\n'

In [72]:
data = {
    'data': [],
    'label': []
}

In [73]:
# Procesar los archivos de entrenamiento en paralelo utilizando múltiples hilos (threads).
# Se usa `ThreadPoolExecutor` para paralelizar el procesamiento y mejorar la velocidad.
with ThreadPoolExecutor(max_workers=os.cpu_count() - 2) as executor:
    # Se aplica la función `process_file_moon` a cada índice en `indexs_train` y se muestra un progreso con tqdm.
    results = list(tqdm(executor.map(process_file, indexs), total=len(indexs)))

# Recopilar los datos procesados de entrenamiento.
for r in results:
    # Si el archivo no pudo procesarse (devuelve `None`), se imprime "skip" y se pasa al siguiente.
    if r[0] is None:
        print('skip')
        continue
    # Se agrega el dato procesado (entrada) a la lista `data['data']`.
    data['data'].append(r[0])
    # Se agrega la etiqueta correspondiente (salida) a la lista `data['label']`.
    data['label'].append(r[1])

  0%|          | 0/920 [00:00<?, ?it/s]

In [76]:
# Convertir enfermedades a números
encoder = LabelEncoder()
data_encoded = encoder.fit_transform(data['label'])

In [77]:
# Convertir las listas en arreglos de numpy:

data['data'] = np.array(data['data'])  # Convierte los datos de entrenamiento en un arreglo numpy
data['label'] = np.array(data_encoded)  # Convierte las etiquetas de entrenamiento en un arreglo numpy

In [78]:
X = data['data']
Y = data['label']

In [79]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Ajustanos mean y std

In [80]:
def train_random_forest_optimized(X, y, test_size=0.2, cv=5):

  """ Entrena un modelo Random Forest optimizado con validación cruzada y búsqueda de hiperparámetros.

  Parámetros:
  - X: Matriz de características.
  - y: Vector de etiquetas.
  - test_size: Proporción de datos para prueba.
  - cv: Número de folds en la validación cruzada.

  Retorna:
  - Mejor modelo entrenado.
  - Diccionario con métricas de evaluación.
  """

  # División en entrenamiento y prueba
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

  # Búsqueda de mejores hiperparámetros con GridSearchCV
  param_grid = {
      'n_estimators': [100, 200, 300],
      'max_depth': [10, 20, None],
      'min_samples_split': [2, 5, 10],
      'min_samples_leaf': [1, 2, 4]
    }

  grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=cv, n_jobs=-1)
  grid_search.fit(X_train, y_train)

  model = grid_search.best_estimator_

  # Evaluación con validación cruzada
  scores = cross_val_score(model, X, y, cv=cv)
  mean_accuracy = np.mean(scores)

  # Evaluación en datos de prueba
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  class_report = classification_report(y_test, y_pred)
  conf_matrix = confusion_matrix(y_test, y_pred)

  print(f"Mejor modelo encontrado: {grid_search.best_params_}")
  print(f"Precisión media (validación cruzada {cv}-fold):{mean_accuracy:.4f}")
  print(f"Accuracy en prueba: {accuracy:.4f}")
  print("Reporte de Clasificación:\n", class_report)
  print("Matriz de Confusión:\n", conf_matrix)

  # Devolver modelo y métricas
  return model, {
      "accuracy_test": accuracy,
      "mean_cv_accuracy": mean_accuracy,
      "classification_report": class_report,
      "confusion_matrix": conf_matrix
  }


In [81]:
def predict_audio(modelo, indice):

  """ Predice la enfermedad en un nuevo archivo de audio basado en su matriz de características.

  Parámetros:
  - modelo: Modelo Random Forest entrenado.
  - audio_path: Ruta del archivo de audio a predecir.

  Retorna:
  - Clase predicha.
  """ 

  features = process_file(indice)
  f_scales = scaler.transform(features)
  prediction = modelo.predict(f_scales)
  return prediction[0]

In [89]:
#Entrenar el modelo de Random Forest
modelo_rf, metrics = train_random_forest_optimized(X_scaled, Y)


c:\Users\jhose\anaconda3\envs\data\lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\jhose\anaconda3\envs\data\lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Mejor modelo encontrado: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Precisión media (validación cruzada 5-fold):0.8859
Accuracy en prueba: 0.8696
Reporte de Clasificación:
               precision    recall  f1-score   support

           1       1.00      0.22      0.36         9
           2       0.00      0.00      0.00         4
           3       0.90      1.00      0.95       150
           4       0.55      0.75      0.63         8
           6       1.00      0.11      0.20         9
           7       0.33      0.25      0.29         4

    accuracy                           0.87       184
   macro avg       0.63      0.39      0.40       184
weighted avg       0.86      0.87      0.83       184

Matriz de Confusión:
 [[  2   0   2   3   0   2]
 [  0   0   3   1   0   0]
 [  0   0 150   0   0   0]
 [  0   0   2   6   0   0]
 [  0   0   8   0   1   0]
 [  0   0   2   1   0   1]]


c:\Users\jhose\anaconda3\envs\data\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\jhose\anaconda3\envs\data\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\jhose\anaconda3\envs\data\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [91]:
#Probar el modelo con un nuevo audio
new_audio = 301
predicted_label = predict_audio(modelo_rf, new_audio)
# Convertir el número a nombre de enfermedad
prediction_label = encoder.inverse_transform(predicted_label)

#Mostrar en pantalla el resultado
print(f"Predicción para {new_audio}: Enfermedad {prediction_label}")


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.